# 🕳️ Valores faltantes: diagnosticar antes de operar

**Taller de Obtención y Preparación de Datos** · batería de práctica del [Visualizador TOPD](https://mati3939.github.io/visualizador-numpy-pandas/)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mati3939/visualizador-numpy-pandas/blob/main/notebooks/06_valores_faltantes.ipynb)

**Objetivos:** diagnosticar nulos con `isnull()`, destapar nulos disfrazados, entender `dropna(thresh=...)` y decidir entre media, mediana y moda para imputar.

🔎 **Apóyate en el visualizador:** [Valores perdidos](https://mati3939.github.io/visualizador-numpy-pandas/#nulos) — ten la pestaña abierta mientras resuelves; cada concepto de aquí está animado allá.

---

## Contexto: plataforma de telemedicina

Trabajas con el registro de consultas de una plataforma de telemedicina regional — el mismo escenario del **Control 3** del curso. Los formularios los llenan médicos apurados entre una videollamada y otra, así que el dataset llega con huecos… y no todos los huecos se ven como huecos.

## 0 · Preparación

Corre esta celda primero (define los datos de todo el notebook).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(314)
n = 50

especialidades = ['Medicina general', 'Pediatría', 'Dermatología', 'Nutrición', 'Psicología']
comunas = ['Concepción', 'Talcahuano', 'San Pedro', 'Hualpén', 'Chiguayante']

consultas = pd.DataFrame({
    'id_consulta': np.arange(1, n + 1),
    'especialidad': rng.choice(especialidades, n),
    'comuna': rng.choice(comunas, n),
    'duracion_min': rng.normal(20, 4, n).round(1).clip(8, None),
    'edad_paciente': rng.integers(18, 90, n).astype(float),
    'satisfaccion': rng.integers(1, 6, n),
})

# 3 consultas con duraciones absurdamente largas (videollamadas que quedaron pegadas)
outliers_idx = [5, 22, 40]
consultas.loc[outliers_idx, 'duracion_min'] = [95.0, 120.0, 88.0]

# nulos DE VERDAD (np.nan), con una proporción distinta por columna
mask_comuna = rng.random(n) < 0.08
mask_duracion = rng.random(n) < 0.14
mask_duracion[outliers_idx] = False   # no tapamos los outliers, los necesitamos enteros
mask_edad = rng.random(n) < 0.22

consultas.loc[mask_comuna, 'comuna'] = np.nan
consultas.loc[mask_duracion, 'duracion_min'] = np.nan
consultas.loc[mask_edad, 'edad_paciente'] = np.nan

# nulos DISFRAZADOS en 'satisfaccion': algunos médicos anotan 'sin dato' o -999
# en vez de dejar el campo vacío — para pandas eso NO es un nulo
consultas['satisfaccion'] = consultas['satisfaccion'].astype(object)
idx_sin_dato = rng.choice(n, size=4, replace=False)
resto = [i for i in range(n) if i not in idx_sin_dato]
idx_menos999 = rng.choice(resto, size=3, replace=False)
consultas.loc[idx_sin_dato, 'satisfaccion'] = 'sin dato'
consultas.loc[idx_menos999, 'satisfaccion'] = -999

consultas.head(10)

## 1 · Calentamiento

Antes de tocar nada, revisa la animación de nulos en [https://mati3939.github.io/visualizador-numpy-pandas/#nulos](https://mati3939.github.io/visualizador-numpy-pandas/#nulos).

### 1.1 Radiografía de nulos ⭐

Calcula `nulos_por_col`, el número de nulos de **cada columna** con `isnull().sum()`, y `pct_nulos`, el mismo dato pero en **porcentaje** (redondeado a 1 decimal).

<details><summary>💡 Pista (haz clic)</summary>

`isnull()` marca `True` donde hay `NaN` — `.sum()` sobre eso cuenta los `True` de cada columna.

</details>

In [ ]:
nulos_por_col = ...   # TU CÓDIGO AQUÍ
pct_nulos = ...

print(nulos_por_col)
print(pct_nulos)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert list(nulos_por_col.index) == list(consultas.columns)
assert nulos_por_col.sum() == consultas.isnull().sum().sum()
assert abs(pct_nulos['edad_paciente'] - nulos_por_col['edad_paciente'] / len(consultas) * 100) < 1e-9
print('✅ ¡Correcto!')

### 1.2 La consulta más incompleta ⭐

Calcula `nulos_por_fila`, el número de campos vacíos **por fila** (`axis=1`), y guarda en `id_mas_incompleta` el `id_consulta` de la fila con más nulos.

<details><summary>💡 Pista (haz clic)</summary>

`idxmax()` sobre `nulos_por_fila` te da la ETIQUETA (el índice) de la fila peor, no el conteo.

</details>

In [ ]:
nulos_por_fila = ...   # TU CÓDIGO AQUÍ
id_mas_incompleta = ...

print(nulos_por_fila.max())
print('id:', id_mas_incompleta)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert nulos_por_fila.shape[0] == len(consultas)
assert nulos_por_fila.max() == consultas.isnull().sum(axis=1).max()
assert id_mas_incompleta in consultas['id_consulta'].values
print('✅ ¡Correcto!')

## 2 · Núcleo

Fíjate en algo raro: en el diagnóstico de arriba, `satisfaccion` salió con **0 nulos**. ¿En serio no le falta nada a esa columna? Ábrela con `.unique()` antes de creerte esa cifra.

### 2.1 Nulos que se disfrazan ⭐⭐

Mira `consultas['satisfaccion'].unique()` y detecta los valores que en realidad representan datos ausentes. Construye `satisfaccion_limpia` reemplazando esos centinelas por `np.nan` con un solo `.replace({...})`, y conviértela a `float`.

<details><summary>💡 Pista (haz clic)</summary>

`isnull()` no detecta `'sin dato'` ni `-999` — para pandas son valores tan válidos como cualquier otro, hasta que tú les digas lo contrario con `replace`.

</details>

In [ ]:
print(consultas['satisfaccion'].unique())

satisfaccion_limpia = ...   # TU CÓDIGO AQUÍ

print(satisfaccion_limpia.isnull().sum())

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert satisfaccion_limpia.dtype.kind == 'f'
assert satisfaccion_limpia.isnull().sum() == consultas['satisfaccion'].isin(['sin dato', -999]).sum()
assert not (satisfaccion_limpia == -999).any()
print('✅ ¡Correcto!')

### 2.2 dropna con exigencia (thresh) ⭐⭐

`thresh` en `dropna` NO es un máximo de nulos permitido: es el número **mínimo de valores NO nulos** que exige por fila. Usa `thresh = consultas.shape[1] - 1` (como máximo un campo vacío por fila) para construir `consultas_filtradas`.

<details><summary>💡 Pista (haz clic)</summary>

Con 6 columnas, `thresh=5` bota solo las filas con 2 o más campos vacíos — no confundas 'mínimo exigido' con 'máximo tolerado'.

</details>

In [ ]:
thresh_min = consultas.shape[1] - 1

consultas_filtradas = ...   # TU CÓDIGO AQUÍ

print(len(consultas), '->', len(consultas_filtradas))

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert len(consultas_filtradas) <= len(consultas)
assert (consultas_filtradas.notnull().sum(axis=1) >= thresh_min).all()
assert len(consultas_filtradas) == (consultas.notnull().sum(axis=1) >= thresh_min).sum()
print('✅ ¡Correcto!')

### 2.3 Mediana vs promedio, ¿a quién le creemos? ⭐⭐

Calcula `media_duracion` y `mediana_duracion` de `duracion_min` (ignoran los `NaN` solas). Luego construye `duracion_imputada` rellenando los nulos con la que consideres más robusta frente a los 3 outliers de la preparación.

<details><summary>💡 Pista (haz clic)</summary>

Los outliers estiran la media hacia ellos; la mediana los ignora casi por completo — por eso se prefiere para rellenar nulos en columnas con outliers.

</details>

In [ ]:
media_duracion = ...      # TU CÓDIGO AQUÍ
mediana_duracion = ...

duracion_imputada = ...   # TU CÓDIGO AQUÍ

print(round(media_duracion, 1), round(mediana_duracion, 1))
print(duracion_imputada.isnull().sum())

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert duracion_imputada.isnull().sum() == 0
assert media_duracion > mediana_duracion, 'con estos outliers la media debería quedar por encima de la mediana'
_hueco = consultas['duracion_min'].isnull()
assert (duracion_imputada[_hueco] == mediana_duracion).all()
print('✅ ¡Correcto!')

### 2.4 La moda salva la categórica ⭐⭐

`comuna` es categórica: ni media ni mediana tienen sentido ahí. Calcula `comuna_moda` con `.mode()[0]` y usa esa moda para construir `comuna_imputada` sin nulos.

<details><summary>💡 Pista (haz clic)</summary>

`mode()` puede devolver más de un valor si hay empate — por eso siempre se pide `[0]`.

</details>

In [ ]:
comuna_moda = ...   # TU CÓDIGO AQUÍ

comuna_imputada = ...   # TU CÓDIGO AQUÍ

print(comuna_moda)
print(comuna_imputada.isnull().sum())

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert comuna_imputada.isnull().sum() == 0
assert comuna_moda in consultas['comuna'].dropna().unique()
assert comuna_moda == consultas['comuna'].mode()[0]
print('✅ ¡Correcto!')

## 3 · Desafío

### 3.1 El reporte antes/después 🏁 ⭐⭐⭐

Arma `consultas_limpia`: aplica a **todas** las columnas con problemas el arreglo que corresponda (el mismo truco de `satisfaccion` para los centinelas, mediana para `duracion_min` y `edad_paciente`, moda para `comuna`). Construye `reporte`, un DataFrame con el % de nulos de cada columna **antes** y **después**, y confirma que no queda ningún nulo.

<details><summary>💡 Pista (haz clic)</summary>

Ojo: `satisfaccion` primero necesita el `replace` del ejercicio 2.1 — recién después de eso tiene sentido calcularle una mediana.

</details>

In [ ]:
pct_antes = (consultas.isnull().sum() / len(consultas) * 100).round(1)

consultas_limpia = consultas.copy()
consultas_limpia['satisfaccion'] = ...   # TU CÓDIGO AQUÍ
consultas_limpia['duracion_min'] = ...
consultas_limpia['edad_paciente'] = ...
consultas_limpia['comuna'] = ...

pct_despues = ...   # TU CÓDIGO AQUÍ

reporte = pd.DataFrame({'antes_%': pct_antes, 'despues_%': pct_despues})
print(reporte)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert consultas_limpia.isnull().sum().sum() == 0, 'no deben quedar nulos'
assert (reporte['despues_%'] == 0).all()
assert reporte.loc['edad_paciente', 'antes_%'] == pct_antes['edad_paciente']
assert len(consultas_limpia) == len(consultas), 'imputar no debe borrar filas'
print('✅ ¡Correcto!')

---

## 🎯 Chequea tu intuición

Antes de cerrar, abre el visualizador y en el botón **🎯 Ejercicios** corre una ronda de [Predice la salida](https://mati3939.github.io/visualizador-numpy-pandas/#predice) y [Detective de bugs](https://mati3939.github.io/visualizador-numpy-pandas/#detective) filtrando por **Nulos**. Si un error de Python te deja pegado, pégalo en el [Traductor de errores](https://mati3939.github.io/visualizador-numpy-pandas/#traductor).